In [2]:
# %load_ext autoreload
# %autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
from typing import Tuple, List, Set

# Add project root to sys.path
project_root = Path('..').resolve()
sys.path.append(str(project_root))

# Import project modules
from src.data_loader.cmapss.CMAPSSDataLoader import CMAPSSDataLoader
from src.data_loader.cmapss.CMAPSSTimeSeriesDataset import CMAPSSTimeSeriesDataset
from src.data_loader.cmapss.CMAPSSDatasetWrapper import CMAPSSDatasetWrapper
from src.models.TransformerModel import TransformerModel
from src.models.LSTMModel import LSTMModel
from src.counterfactuals.core import BasisGenerator  
from src.counterfactuals.basis import BSplineBasis

from src.data_loader.cmapss.CMAPSSTorchDataloader import CMAPSSTorchDataloader

from torch.utils.data import DataLoader

from src.counterfactuals.utils.cmapss.load_data import load_and_preprocess_data
from src.counterfactuals.utils.cmapss.cf_utils import get_unit_sequences, predict_rul, get_valid_target_rul, get_last_sequence


# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device("cpu")
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)


Using device: cpu


# Load Data

In [3]:
dataloader = CMAPSSTorchDataloader()
data_subset = 'FD001'

train_loader, val_loader, test_loader, preprocess, train_norm, test_norm, val_norm = load_and_preprocess_data(
            subset="FD001",
            seq_len=50,
            max_rul=125,
            batch_size=32,
            seed=42,
            val_ratio=0.2,
            var_threshold=0.01,
        )

feature_cols = preprocess['feature_cols']


Loading + Preprocessing FD001
Loaded FD001:
  Training samples: 20631, Units: 100
  Test samples: 13096, Units: 100
Removed 10 low-variance features: ['setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Kept 14 features

Final feature_dim = 14
Train batches = 402 | Val batches = 90 | Test batches = 3


# Check the data dist

In [4]:
# Quick sanity check: label distributions
import numpy as np

train_labels = []
for _, y in train_loader:
    train_labels.extend(y.numpy().flatten().tolist())

val_labels = []
for _, y in val_loader:
    val_labels.extend(y.numpy().flatten().tolist())

train_labels = np.array(train_labels)
val_labels = np.array(val_labels)

print(f"\n{'='*60}")
print("LABEL DISTRIBUTION SANITY CHECK")
print(f"{'='*60}")
print(f"  Train labels — min: {train_labels.min():.4f}, max: {train_labels.max():.4f}, "
      f"mean: {train_labels.mean():.4f}, std: {train_labels.std():.4f}")
print(f"  Val   labels — min: {val_labels.min():.4f}, max: {val_labels.max():.4f}, "
      f"mean: {val_labels.mean():.4f}, std: {val_labels.std():.4f}")
print(f"{'='*60}\n")


LABEL DISTRIBUTION SANITY CHECK
  Train labels — min: 0.0000, max: 125.0000, mean: 76.3899, std: 41.5680
  Val   labels — min: 0.0000, max: 125.0000, mean: 70.5947, std: 40.2492



# Load the Model

In [5]:
sequence_length = 50
input_size = len(feature_cols)

model = TransformerModel(
    input_size=input_size,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.2
).to(device)

model_path = '/home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/saved_models/transformer_best_last.pth'
model_checkpoint_path = '/home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/saved_models/transformer_best_last.ckpt'

if os.path.exists(model_path):
    try:
        checkpoint = torch.load(model_checkpoint_path, map_location=device)
        print(f"{checkpoint.keys()}")
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        print(f"✅ Loaded model from {model_path}")
    except Exception as e:
        print(f"⚠️ Model load failed: {e}")
        raise
else:
    print(f"⚠️ Model file not found at {model_path}. Please ensure the model is trained and saved correctly.")

model.eval()
#print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'best_score', 'best_epoch', 'history', 'config', 'preprocess'])
✅ Loaded model from /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/saved_models/transformer_best_last.pth

Total parameters: 275,201


In [6]:
import json

scaler_path = '../outputs/preprocess.json'

if os.path.exists(scaler_path):
    with open(scaler_path, 'r') as f:
        scaler_data = json.load(f)
    norm_stats = {
        'mean': np.array([scaler_data['mean'][col] for col in feature_cols]),
        'std': np.array([scaler_data['std'][col] for col in feature_cols])
    }
    print(f"✅ Loaded normalization stats from {scaler_path}")
else:
    # Use stats from current normalization
    norm_stats = {
        'mean': preprocess['mean'],
        'std': preprocess['std']
    }
    print("⚠️ Using normalization stats from current run")

✅ Loaded normalization stats from ../outputs/preprocess.json


In [8]:
## Basis Generators
generator_fourier = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='fourier',
        num_basis=10,
        device=device,
        normalization_stats=norm_stats
    )

generator_bspline = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='bspline',
        num_basis=10,
        device=device,
        normalization_stats=norm_stats
    )

generator_wavelet = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='wavelet',
        num_basis=10,
        device=device,
        normalization_stats=norm_stats
    )

generator_polynomial = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='polynomial',
        num_basis=10,
        device=device,
        normalization_stats=norm_stats
    )

generator_rbf = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='rbf',
        num_basis=10,
        device=device,
        normalization_stats=norm_stats
    )

generators = {"fourier":generator_fourier, 
              "bspline":generator_bspline, 
              "wavelet":generator_wavelet, 
              "polynomial":generator_polynomial, 
              "rbf":generator_rbf
            }

print("\n✅ Basis generators initialized successfully")


✅ Basis generators initialized successfully


In [ ]:
def generate_cfs_for_multiple_units(df, unit_ids, model, generator, 
                                     sequence_length, feature_cols, 
                                     increase_range=(30, 50), max_rul=125):
    """Generate counterfactuals for multiple units"""
    results = []
    
    for unit_id in unit_ids:
        query_seq, true_rul = get_last_sequence(df, unit_id, sequence_length, feature_cols)
        
        if query_seq is None:
            continue
        
        query_instance = torch.tensor(query_seq, dtype=torch.float32).to(device)
        
        try:
            current_pred = predict_rul(model, query_seq, device)
            target_rul = get_valid_target_rul(current_pred, increase_range=increase_range, max_rul=max_rul)
            
            cfs = generator.generate(
                query_instance=query_instance,
                target_rul=target_rul,
                num_cfs=1,
                lr=0.001,
                max_iter=1000,
                lambdas={
                    'validity': 10.0,
                    'prox': 9.0,
                    'sparsity': 0.5,
                    'diversity': 0.0,
                    'smoothness': 1.3,
                },
                verbose=False
            )
            
            # Convert counterfactual back to numpy for prediction
            cf_numpy = cfs[0].detach().cpu().numpy()
            cf_pred = predict_rul(model, cf_numpy, device)
            
            results.append({
                'unit_id': unit_id,
                'true_rul': true_rul,
                'original_pred': current_pred,
                'target_rul': target_rul,
                'cf_pred': cf_pred,
                'success': abs(cf_pred - target_rul) < 10.0
            })
            
            print(f"Unit {unit_id}: Original={current_pred:.1f}, Target={target_rul:.1f}, CF={cf_pred:.1f}, Δ={abs(cf_pred - target_rul):.1f}")
            
        except Exception as e:
            print(f"Failed for unit {unit_id}: {e}")
            import traceback
            traceback.print_exc()
    
    return pd.DataFrame(results)

# Generate for first 10 test units
test_unit_ids = list(range(1, 11))
results_df = generate_cfs_for_multiple_units(
    test_norm, test_unit_ids, model, generator, 
    sequence_length, feature_cols, increase_range=(10, 30), max_rul=125
)

print(f"\nSuccess Rate: {results_df['success'].mean()*100:.1f}%")
results_df